In [1]:
import sys
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add dirctory with model moduels to path
sys.path.insert(0, os.path.join(os.getcwd(),'..'))

import CIBUSmod as cm
from CIBUSmod.utils.output_data_manip import concat_herds

# Create input data
x0_crp.csv and x0_ani.csv represent the current areas of different crops and number of heads of different animals respectively. These are both still WIP, which is why some mangling is needed here to get them in the right form.
The demand vector is generated with demand for cattle meat, pig meat and cattle milk equal to what is in the agricultural statistics.

In [2]:
# Define x0_crp
x0_crp = \
    pd.read_csv(os.path.join('..','data','x0','x0_crp.csv'), dtype={'region': object})\
    .set_index(['crop','prod_system','region'])['area']

# Define x0_ani
x0_ani = pd.read_csv(os.path.join('..','data','x0','x0_ani.csv'), dtype={'region': object})

x0_ani['species'] = np.nan
x0_ani['species'] = \
np.where(np.isin(x0_ani['animal'], ['kor för mjölkproduktion', 'kor för uppfödning av kalvar']),'cattle',x0_ani['species'])
x0_ani['species'] = \
np.where(x0_ani['animal']=='suggor för avel','pigs',x0_ani['species'])

x0_ani['breed'] = np.nan
x0_ani['breed'] = \
np.where(x0_ani['animal']=='kor för mjölkproduktion','dairy',x0_ani['breed'])
x0_ani['breed'] = \
np.where(x0_ani['animal']=='kor för uppfödning av kalvar','beef',x0_ani['breed'])
x0_ani['breed'] = \
np.where(x0_ani['animal']=='suggor för avel','none',x0_ani['breed'])

x0_ani = x0_ani[x0_ani['species']!='nan'][['species','breed','prod_system','region','number']].set_index(['species','breed','prod_system','region'])['number']

x0_ani = x0_ani.fillna(0)

x0 = {'ani':x0_ani,'crp':x0_crp}

In [3]:
# Create demand vectors
# Animal products
D_ani = pd.Series(
    {
        ('conventional','cattle','meat') : (136-21) * 1000000,
        ('organic','cattle','meat')      : 21 * 1000000,
        ('conventional','cattle','milk') : (2760-464) * 1000000,
        ('organic','cattle','milk')      : 464 * 1000000,
        ('conventional','pigs','meat')   : (249-6) * 1000000,
        ('organic','pigs','meat')        : 6 * 1000000,
    },
)
D_ani.index.rename(['prod_system','species','product'], inplace=True)

# Crop products (kg DM)
D_crp = pd.Series(
    {
        ('conventional','wheat')         :0,
        ('organic','wheat')              :0,
    },
)
D_crp.index.rename(['prod_system','crop_product'], inplace=True)

D = {'ani':D_ani, 'crp':D_crp}

# Run base year
This runs the model for the base year (2016-2020). Results are then compared to statistics to ensure feasible results replicating the current situation.

In [4]:
tic = time.time()

# Instantiate crop production
crops = cm.CropProduction(
    par = cm.ParameterRetriever(
        os.path.join('..','data','prod_parameters','CropProduction.xlsx')
    ),
    index = x0_crp.index
)    

# Instantiate animal herds
herds=pd.Series(
    data=[],
    index=pd.MultiIndex(
        levels=[[]]*4,
        codes=[[]]*4,
        names=['species','breed','prod_system','sub_system']
    ),
    dtype = object
)

for (sp,br,ps) in x0_ani.groupby(['species','breed','prod_system']).sum().index:
            
    if sp == 'cattle':
        herds[(sp,br,ps,'none')] = \
            cm.CattleHerd(
                par = cm.ParameterRetriever(
                    os.path.join('..','data','prod_parameters','CattleHerd.xlsx')
                ),
                index = x0_ani.index.get_level_values('region').unique(),
                breed = br,
                prod_system = ps
            )

    elif sp == 'pigs':
        herds[(sp,br,ps,'none')] = \
            cm.PigHerd(
                par = cm.ParameterRetriever(
                    os.path.join('..','data','prod_parameters','PigHerd.xlsx')
                ),
                index = x0_ani.index.get_level_values('region').unique(),
                breed = br,
                prod_system = ps
            ) 

# Instantiate manure management
feed_mgmt = cm.FeedMgmt(
    herds = herds,
    par = cm.ParameterRetriever(
        os.path.join('..','data','prod_parameters','FeedMgmt.xlsx')
    )
)

# Instantiate manure management
manure_mgmt = cm.ManureMgmt(
    herds = herds,
    par = cm.ParameterRetriever(
        os.path.join('..','data','prod_parameters','ManureMgmt.xlsx')
    )
)

# Instantiate geo distributor
geodist = cm.GeoDistributor(D,x0,crops,herds,feed_mgmt)

# -------------------------------------------------------------------- #

# Calculate crops
crops.calculate(
    verbose = True
)

# Calculate herds
for h in herds:
    h.calculate(verbose = True)

# Calculate feed
feed_mgmt.calculate(verbose=True)    

# Calculate manure
manure_mgmt.calculate(verbose=True)

# Distribute animals and crops
geodist.make(use_cons=[1,2,3,4],verbose=True)
geodist.solve(verbose=True)

# Scale and store results
out_crops = crops.scale(geodist.x['crp'])

out_animals = concat_herds([
    h.scale(
        geodist.x['ani'].loc[(h.species,h.breed,h.prod_system,h.sub_system)],
        x_is = h.x_is
    )
    for h in herds
])

print(f'Done! Elapsed time: {(time.time()-tic):.0f} sec')

[13:39:49][CropProduction] Calculating harvest ...
[13:39:49][CropProduction] Calculating production ...


C:\Users\jnka0003\Documents\#1 Projekt\MISTRA Food Futures WP5\CIBUSmod Food Systems Model\notebooks\..\CIBUSmod\utils\retriever.py:163: UserWarning: NaNs returned! No value for 'yield' found in '..\data\prod_parameters\CropProduction.xlsx' for some of the supplied filters (n=503): 
----------
f_crop = Beans, dry 
f_prod_system = conventional
f_region = 1011
----------
f_crop = Beans, dry 
f_prod_system = conventional
f_region = 111
----------
f_crop = Beans, dry 
f_prod_system = conventional
f_region = 1111
----------
f_crop = Beans, dry 
f_prod_system = conventional
f_region = 1112
----------
f_crop = Beans, dry 
f_prod_system = conventional
f_region = 112
 ...
  warnings.warn(str1+str2)


[13:39:49][CropProduction] Done! Elapsed time: 0 sec
[13:39:49][AnimalHerd (cattle, beef, conventional, none)] Calculating herd structure ...
[13:39:49][AnimalHerd (cattle, beef, conventional, none)] Calculating production ...
[13:39:49][AnimalHerd (cattle, beef, conventional, none)] Done! Elapsed time: 1 sec
[13:39:49][AnimalHerd (cattle, beef, organic, none)] Calculating herd structure ...
[13:39:50][AnimalHerd (cattle, beef, organic, none)] Calculating production ...
[13:39:50][AnimalHerd (cattle, beef, organic, none)] Done! Elapsed time: 1 sec
[13:39:50][AnimalHerd (cattle, dairy, conventional, none)] Calculating herd structure ...
[13:39:51][AnimalHerd (cattle, dairy, conventional, none)] Calculating production ...
[13:39:51][AnimalHerd (cattle, dairy, conventional, none)] Done! Elapsed time: 0 sec
[13:39:51][AnimalHerd (cattle, dairy, organic, none)] Calculating herd structure ...
[13:39:51][AnimalHerd (cattle, dairy, organic, none)] Calculating production ...
[13:39:51][AnimalHe

C:\Users\jnka0003\Documents\#1 Projekt\MISTRA Food Futures WP5\CIBUSmod Food Systems Model\notebooks\..\CIBUSmod\utils\retriever.py:151: UserWarning: NaNs returned! No value for 'slaughter_weight' found in '..\data\prod_parameters\PigHerd.xlsx' matching any of the supplied filters (n=1): 
----------
f_prod_system = conventional
f_animal = piglets
  warnings.warn(str1+str2)
C:\Users\jnka0003\Documents\#1 Projekt\MISTRA Food Futures WP5\CIBUSmod Food Systems Model\notebooks\..\CIBUSmod\utils\retriever.py:151: UserWarning: NaNs returned! No value for 'slaughter_weight' found in '..\data\prod_parameters\PigHerd.xlsx' matching any of the supplied filters (n=1): 
----------
f_prod_system = conventional
f_animal = gilts
  warnings.warn(str1+str2)
C:\Users\jnka0003\Documents\#1 Projekt\MISTRA Food Futures WP5\CIBUSmod Food Systems Model\notebooks\..\CIBUSmod\utils\retriever.py:151: UserWarning: NaNs returned! No value for 'slaughter_weight' found in '..\data\prod_parameters\PigHerd.xlsx' match

[13:39:52][AnimalHerd (pigs, none, conventional, none)] Done! Elapsed time: 0 sec
[13:39:52][AnimalHerd (pigs, none, organic, none)] Calculating herd structure ...
[13:39:52][AnimalHerd (pigs, none, organic, none)] Calculating production ...
[13:39:52][AnimalHerd (pigs, none, organic, none)] Done! Elapsed time: 0 sec
[13:39:52][FeedMgmt] Calculating feed consumption ...


C:\Users\jnka0003\Documents\#1 Projekt\MISTRA Food Futures WP5\CIBUSmod Food Systems Model\notebooks\..\CIBUSmod\utils\retriever.py:151: UserWarning: NaNs returned! No value for 'live_weight' found in '..\data\prod_parameters\CattleHerd.xlsx' matching any of the supplied filters (n=1): 
----------
f_breed = beef
f_prod_system = conventional
f_sub_system = none
f_animal = breeding bulls
  warnings.warn(str1+str2)
C:\Users\jnka0003\Documents\#1 Projekt\MISTRA Food Futures WP5\CIBUSmod Food Systems Model\notebooks\..\CIBUSmod\utils\retriever.py:151: UserWarning: NaNs returned! No value for 'live_weight' found in '..\data\prod_parameters\CattleHerd.xlsx' matching any of the supplied filters (n=1): 
----------
f_breed = beef
f_sub_system = none
f_prod_system = organic
f_animal = breeding bulls
f_from_ps = organic
f_to_ps = conventional
  warnings.warn(str1+str2)
C:\Users\jnka0003\Documents\#1 Projekt\MISTRA Food Futures WP5\CIBUSmod Food Systems Model\notebooks\..\CIBUSmod\utils\retriever.p

[13:39:56][FeedMgmt] Calculating demand for crop products ...
[13:39:58][FeedMgmt] Calculating demand for by-products ...
[13:39:59][FeedMgmt] Done! Elapsed time: 7 sec
[13:39:59][ManureMgmt] Calculating N excretion ...
[13:40:00][ManureMgmt] Calculating N losses ...
[13:40:05][ManureMgmt] Calculating N available to spread ...
[13:40:05][ManureMgmt] Done! Elapsed time: 6 sec
[13:40:05][GeoDist.make] Making constraint C1 ...
[13:40:07][GeoDist.make] Making constraint C2 ...
[13:41:08][GeoDist.make] Making constraint C3 ...
[13:41:08][GeoDist.make] Making constraint C4 ...
[13:41:08][GeoDist.make] Making objective O1 ...
[13:41:08][GeoDist.make] Defining problem ...
[13:41:08][GeoDist.make] Done! Elapsed time: 63 sec
[13:41:08][GeoDist.solve] Finding solution ...
[13:41:17][GeoDist.solve] Optimal solution found! Status: 'optimal', Solver: 'OSQP'
[13:41:17][GeoDist.solve] Done! Elapsed time: 9 sec
Done! Elapsed time: 91 sec


In [15]:
geodist.solve(
    verbose=True,
    solver_settings = {
        'solver':'OSQP',
        'verbose':True,
        'max_iter':200000
    }
)

[13:47:09][GeoDist.solve] Finding solution ...
                                     CVXPY                                     
                                     v1.3.0                                    
(CVXPY) Apr 21 01:47:09 PM: Your problem has 12084 variables, 4 constraints, and 0 parameters.
(CVXPY) Apr 21 01:47:09 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Apr 21 01:47:09 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Apr 21 01:47:09 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Apr 21 01:47:09 PM: Compiling problem (target solver=OSQP).
(CVXPY) Apr 21 01:47:09 PM: Reduction chain: C

In [5]:
geodist.problem.value

1292212928.8519626

In [7]:
idx = pd.IndexSlice

print(
    pd.concat([
        out_animals.heads
        .groupby(['species','breed','prod_system','animal'], axis=1).sum()
        .sum().loc[idx[:,:,:,['cows','sows']]].droplevel('animal')
        .rename('x')
        ,
        x0_ani
        .groupby(['species','breed','prod_system']).sum()
        .rename('x0')
    ], axis=1)
    .apply(
        lambda x:
        pd.Series(
            [x['x'].round(1),x['x0'].round(1),((x['x']-x['x0'])/x['x0']*100).round(1)],
            index = ['x','x0','% dif']
        ),
        axis=1
    )
)

print(
    pd.concat([
        out_crops.area.groupby('crop').sum().rename('x'),
        x0_crp.groupby('crop').sum().rename('x0')
    ], axis=1)
    .apply(
        lambda x:
        pd.Series(
            [x['x'].round(1),x['x0'].round(1),((x['x']-x['x0'])/x['x0']*100).round(1)],
            index = ['x','x0','% dif']
        ),
        axis=1
    )
)

                                   x        x0  % dif
species breed prod_system                            
cattle  beef  conventional  151637.6  128080.1   18.4
              organic        54762.2   72274.4  -24.2
        dairy conventional  246516.6  263227.4   -6.3
              organic        54567.6   54920.6   -0.6
pigs    none  conventional  101749.5  132262.5  -23.1
              organic         3452.4    3325.0    3.8
                                            x        x0   % dif
crop                                                           
Apples                                 3068.2    1552.0    97.7
Barley, spring                       297268.3  296954.7     0.1
Barley, winter                        19620.5   18442.5     6.4
Beans, dry                             2374.2     703.3   237.6
Broad beans and horse beans, dry      26090.2   25063.8     4.1
Cabbages, cauliflowers and broccoli    2898.1    1468.5    97.4
Carrots                                3208.0    1760.5 